In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# Load the necessary files
efficient_frontier_df   = pd.read_csv("efficient_frontier.tsv", sep="\t")
estimated_returns_df    = pd.read_csv("estimated_returns.tsv", sep="\t")
estimated_covariance_df = pd.read_csv("estimated_covariance.tsv", sep="\t", index_col=0)

# Extract asset tickers from the 'asset' column
assets                  = estimated_returns_df['asset'].tolist()
estimated_returns       = estimated_returns_df.set_index('asset').loc[assets, 'estimated_return'].values
estimated_covariance    = estimated_covariance_df.loc[assets, assets].values

# Function to compute minimum risk for a given target return using scipy.optimize
def compute_min_risk(target_return, estimated_returns, estimated_covariance):
    n = len(estimated_returns)
    x0 = np.ones(n) / n  # initial guess
    bounds = [(0, 1) for _ in range(n)]
    constraints = [
        {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
        {'type': 'eq', 'fun': lambda x: np.dot(x, estimated_returns) - target_return}
    ]
    result = minimize(lambda x: np.dot(x.T, np.dot(estimated_covariance, x)),
                      x0, method='SLSQP', bounds=bounds, constraints=constraints)
    if result.success:
        min_variance = result.fun
        return np.sqrt(min_variance)
    else:
        return np.nan

# Compare each portfolio's risk to the theoretical minimum risk
results = []
for _, row in efficient_frontier_df.iterrows():
    target_return       = row['return']
    actual_risk         = row['risk']
    min_risk            = compute_min_risk(target_return, estimated_returns, estimated_covariance)
    results.append({
        "index": row['index'],
        "target_return": target_return,
        "actual_risk": actual_risk,
        "min_risk": min_risk,
        "is_min_risk": np.isclose(actual_risk, min_risk, atol=1e-4)
    })

# Convert results to DataFrame and save
results_df = pd.DataFrame(results)
results_df.to_csv("efficient_frontier_validation.tsv", sep="\t", index=False)

print("Validation complete. Results saved to efficient_frontier_validation.tsv.")

Validation complete. Results saved to efficient_frontier_validation.tsv.


In [3]:
results_df

,index,target_return,actual_risk,min_risk,is_min_risk
0,0.0,0.023923,0.031198,0.031198,True
1,1.0,0.025020,0.031173,0.031173,True
2,2.0,0.026117,0.031460,0.031460,True
3,3.0,0.027213,0.032439,0.032439,True
4,4.0,0.028310,0.032911,0.032911,True
...,...,...,...,...,...
96,96.0,0.129196,0.317207,0.317207,True
97,97.0,0.130292,0.320937,0.320937,True
98,98.0,0.131389,0.324669,0.324669,True
99,99.0,0.132485,0.328402,0.328402,True
